# Session 06, Part 3 — A complete multi-agent Research Brief Agent

One lead agent and two kinds of subagents, built with Deep Agents, with a fix for every problem of the session's
last section:

| Problem | Mechanism in this notebook |
|---|---|
| Vague task descriptions → duplicated or missing work | the lead's prompt requires Objective · Output · Sources · Boundaries in every `task` |
| Long, unstructured results flood the lead | each researcher returns a short summary in a fixed format: claim, exact quote, url, then gaps |
| Evidence lost when results pass from agent to agent | researchers save `/evidence/<slug>.md`; the verifier reads the files |
| Slow sequential research | several `task` calls in one message run in parallel |
| Cost runs away | per-agent limits on model calls, searches and `task` calls; a usage table per agent |
| A subagent crashes or hangs | a `guard` middleware turns errors and timeouts into a gap the lead reports |
| Unchecked errors spread | the guard checks each researcher result (quote + url per claim) before the lead uses it |
| Sources disagree / wrong final claims | `## Conflicts` keeps both claims; a `verifier` with a fresh context checks the draft against the evidence |

<img src="images/b6_brief_agent.png" width="640">

Docs: [Deep Agents subagents](https://docs.langchain.com/oss/python/deepagents/subagents) ·
[Custom middleware](https://docs.langchain.com/oss/python/langchain/middleware/custom) ·
[Anthropic — multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system)

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

# ── Keys and models (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))  # lead, researchers and verifier
logger.info("Model: %s", model.model_name)

OUT = Path("output")  # the final brief goes here (git-ignored)
OUT.mkdir(exist_ok=True)

from utils import internet_search, prompt, short, stream  # notebooks/utils.py

INFO | Model: gpt-6-luna


## 1. A search tool we can break on purpose

To show partial failure and timeout without waiting for a real outage, the researchers use a thin wrapper around
`internet_search`: a query containing `FAULTS["fail"]` raises an error, one containing `FAULTS["hang"]` never
returns. Both are off unless a cell sets them.

In [2]:
import asyncio

from langchain.tools import tool

FAULTS = {"fail": None, "hang": None}


@tool
async def search(query: str) -> list[dict]:
    """Search the web and return title, url and a short snippet for the 3 best results."""
    if FAULTS["fail"] and FAULTS["fail"] in query.lower():
        raise RuntimeError("search backend unavailable")
    if FAULTS["hang"] and FAULTS["hang"] in query.lower():
        await asyncio.sleep(600)  # far longer than the guard's timeout
    return await internet_search.ainvoke({"query": query})

## 2. The researcher: short cited summary, evidence file, its own budget

Like Claude Code and Codex subagents, the researcher returns **text**: its last message becomes the result of
`task`. The task description and `prompts/brief_researcher.md` fix the format (one line per claim: claim, exact
quote, url; then gaps), so the lead gets a few hundred tokens instead of the raw search results. The raw evidence
goes to a file: with the default `StateBackend`, files a subagent writes are merged into the lead's state, where
the verifier can read them. (`response_format` would force JSON instead; useful when code, not a model, reads it.)

Budget per researcher run: at most 5 model calls (`ModelCallLimitMiddleware`, then it ends with what it has) and
2 searches (`ToolCallLimitMiddleware`).

In [3]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware

researcher = {
    "name": "researcher",
    "description": "Answers ONE sub-question on the web; saves its evidence file and returns cited findings.",
    "system_prompt": prompt("brief_researcher"),
    "tools": [search],
    "model": model,
    "middleware": [
        ModelCallLimitMiddleware(run_limit=5, exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="search", run_limit=2, exit_behavior="continue"),
    ],
}

## 3. The verifier: a fresh context that checks the draft against the evidence

The verifier is an ordinary (isolated) subagent: it sees only the draft the lead sends it and the evidence files,
not the lead's reasoning, so it is not biased toward the draft (the reason Claude Code and Cognition use a separate
reviewer). `tools=[]` removes the search tool; it can still read files, because `read_file` belongs to the Deep
Agents harness.

In [4]:
verifier = {
    "name": "verifier",
    "description": "Checks a draft brief against the files in /evidence/; flags unsupported or conflicting claims. "
                   "Call once, with the full draft in the task description.",
    "system_prompt": prompt("verifier"),
    "tools": [],
    "model": model,
}

## 4. The guard: check each result, turn a failed or hung subagent into a gap

`guard` wraps every `task` call of the lead and does two jobs:
- **Check each sub-result before the lead uses it** (the "validation bottleneck" of Google's scaling study): every
  claim line of a researcher must carry an exact quote and a url; lines without them are removed, and the lead is
  told how many. This is plain code, no model call.
- **Contain failures**: an exception inside a subagent's tool travels up through `task` and would stop the whole
  run; a timeout or an error becomes a `ToolMessage` that tells the lead to report a gap. `GraphBubbleUp` (used by
  `interrupt()` for human-in-the-loop) is re-raised.

In [5]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langgraph.errors import GraphBubbleUp
from langgraph.types import Command

import re

TASK_TIMEOUT = 60  # seconds; a researcher run normally ends in 10-25 s
CLAIM = re.compile(r'^- .+ [—–-] ["“].+["”] \[\S+\]$')  # "- <claim> — "<exact sentence>" [<url>]", the format of brief_researcher.md


def check_result(text: str) -> str:
    """Keep claim lines that have a quote and a url; drop the others and say how many were dropped."""
    lines = text.splitlines()
    claims = [line for line in lines if line.startswith("- ")]
    kept = [line for line in claims if CLAIM.match(line.strip())]
    rest = [line for line in lines if not line.startswith("- ")]
    logger.info("check: %d claim(s) kept, %d dropped (no quote or url)", len(kept), len(claims) - len(kept))
    note = f"CHECKED: {len(kept)} claim(s) kept, {len(claims) - len(kept)} dropped (no quote or url)."
    return "\n".join(kept + rest + [note])


@wrap_tool_call
async def guard(request, handler):
    if request.tool_call["name"] != "task":
        return await handler(request)
    try:
        result = await asyncio.wait_for(handler(request), TASK_TIMEOUT)
        if request.tool_call["args"].get("subagent_type") == "researcher":
            # task returns a Command (messages + the files the subagent wrote) or a plain ToolMessage
            for m in result.update.get("messages", []) if isinstance(result, Command) else [result]:
                if isinstance(m, ToolMessage):
                    m.content = check_result(m.text)
        return result
    except GraphBubbleUp:
        raise
    except Exception as exc:  # timeout or crash of one subagent: report it, keep the run alive
        reason = f"timed out after {TASK_TIMEOUT} s" if isinstance(exc, TimeoutError) else repr(exc)
        logger.warning("subagent %s failed: %s", request.tool_call["args"].get("subagent_type"), reason)
        return ToolMessage(f"SUBAGENT FAILED ({reason}). Report this sub-question under Gaps; do not invent facts.",
                           tool_call_id=request.tool_call["id"], name="task", status="error")

The check on a made-up researcher reply (no model call): the second claim has no quote and no url, so it is removed
before the lead ever sees it.

In [6]:
sample = """- Cold-climate heat pumps keep heating below freezing — "Cold-climate models are designed to provide greater capacity below 32°F." [https://www.energy.gov/example]
- Heat pumps work at any temperature
Gaps: no COP values below -20°C"""
print(check_result(sample))

INFO | check: 1 claim(s) kept, 1 dropped (no quote or url)


- Cold-climate heat pumps keep heating below freezing — "Cold-climate models are designed to provide greater capacity below 32°F." [https://www.energy.gov/example]
Gaps: no COP values below -20°C
CHECKED: 1 claim(s) kept, 1 dropped (no quote or url).


## 5. The lead

The lead plans, delegates in parallel, asks the verifier, and writes the brief. Its own budget: at most 5 `task`
calls per run (3 researchers + 1 verifier + 1 spare).

In [7]:
from deepagents import create_deep_agent

lead = create_deep_agent(
    name="lead",
    model=model,
    subagents=[researcher, verifier],
    system_prompt=prompt("lead"),
    middleware=[guard, ToolCallLimitMiddleware(tool_name="task", run_limit=5, exit_behavior="continue")],
)

## 6. Run: three sub-questions, one search backend down

The request has three independent parts. We break the search for "solar" queries to see partial failure inside a
full run: the other two researchers finish, the lead puts the solar part under Gaps.

In [8]:
REQUEST = ("Brief for a family planning to electrify their home: (1) how well air-source heat pumps work in "
           "freezing weather, (2) how long electric-car batteries last, (3) the payback time of rooftop solar.")

FAULTS.update(fail="solar", hang=None)
state, usage, seconds = await stream(lead, REQUEST, main="lead", show_tokens=False)
FAULTS.update(fail=None)

usage.table(f"Full run: {seconds:.0f} s")

[lead] → task({"description": "Objective: Research how well air-source heat pumps work in freezing weath…)
[lead] → task({"description": "Objective: Research how long electric-car batteries last, including evide…)
[lead] → task({"description": "Objective: Research the payback time for rooftop solar for a household, i…)
[researcher] → search({"query": "site:energy.gov electric vehicle battery lifespan degradation 8 years 100,000 m…)
[researcher] → search({"query": "site:energy.gov air source heat pumps cold climates freezing temperatures perfo…)
[researcher] → search({"query": "site:energy.gov rooftop solar payback period factors electricity rates incentiv…)


WARNING | subagent researcher failed: RuntimeError('search backend unavailable')


[lead] ← task: SUBAGENT FAILED (RuntimeError('search backend unavailable')). Report this sub-question und…


INFO | internet_search('site:energy.gov air source heat pumps cold climates freezing temperat…) -> ok 3587 ms


[researcher] ← search: [{"title": "Field Performance of Inverter-Driven Heat Pumps in Cold ...", "url": "https://…


...

[researcher] ← search: [{"title": "EV Battery Health: Key Findings from 22,700 Vehicle Data Analysis | Geotab", "…
[researcher] → write_file({"file_path": "/evidence/heat-pump-freezing.md", "content": "https://www1.eere.energy.gov/…)
[researcher] ← write_file: Updated file /evidence/heat-pump-freezing.md
[researcher] → write_file({"file_path": "/evidence/ev-battery-life.md", "content": "https://afdc.energy.gov/case/278…)
[researcher] ← write_file: Updated file /evidence/ev-battery-life.md


INFO | check: 3 claim(s) kept, 0 dropped (no quote or url)


[lead] ← task: - Some air-source heat pumps can operate below 0°F, and some do so without electric-resist…


INFO | check: 2 claim(s) kept, 0 dropped (no quote or url)


[lead] ← task: - EV battery packs typically take 10 years or more and well over 100,000 miles before star…
[lead] → task({"description": "Check this draft research brief against the evidence files in /evidence/.…)
[verifier] → ls({"path": "/evidence"})
[verifier] → glob({"pattern": "*", "path": "/evidence"})
[verifier] ← ls: ['/evidence/ev-battery-life.md', '/evidence/heat-pump-freezing.md']
[verifier] ← glob: ['/evidence/ev-battery-life.md', '/evidence/heat-pump-freezing.md']
[verifier] → read_file({"file_path": "/evidence/ev-battery-life.md", "offset": 0, "limit": 200})
[verifier] → read_file({"file_path": "/evidence/heat-pump-freezing.md", "offset": 0, "limit": 200})
[verifier] ← read_file: @@ lines 1-2 of 2 @@ https://afdc.energy.gov/case/2788?text= | Well, we now know that it t…
[verifier] ← read_file: @@ lines 1-3 of 3 @@ https://www1.eere.energy.gov/buildings/publications/pdfs/building_ame…
[lead] ← task: Some air-source heat pumps can operate below 0°F, and many do so without e

What happened, read from the final state. Whether the lead sends the three `task` calls in one message is the model's
choice on each run (the prompt asks for it); if the first line shows `[1, 1, 1, ...]`, the dispatch was sequential:
rerun the run cell to see the parallel case.

In [9]:
messages = state["messages"]
dispatch = [len([c for c in m.tool_calls if c["name"] == "task"]) for m in messages if getattr(m, "tool_calls", None)]
print("task calls per lead message:", dispatch, "→ the researchers were dispatched in parallel" if max(dispatch) > 1 else "")

first_task = next(c for m in messages for c in getattr(m, "tool_calls", []) or [] if c["name"] == "task")
print("\nOne task description written by the lead:\n" + first_task["args"]["description"])

print("\nWhat the lead received from each researcher, after the guard's check:")
for m in messages:
    if isinstance(m, ToolMessage) and m.name == "task" and "VERDICT" not in m.text:
        last = m.text.splitlines()[-1]
        print(f"  {short(m.text.splitlines()[0], 110)}\n     … {last}")

print("\nevidence files in the lead's state:", sorted(p for p in state.get("files", {}) if p.startswith("/evidence/")))

task calls per lead message: [3, 1] → the researchers were dispatched in parallel

One task description written by the lead:
Objective: Research how well air-source heat pumps work in freezing weather for a family considering home electrification. Output: findings with each claim, URL, exact supporting quote, and gaps; end with a CHECKED line that identifies only claims verified against cited evidence. Sources: Prefer government agencies, research institutes, and manufacturers. Boundaries: Only cold-weather performance of air-source heat pumps; save evidence to /evidence/heat-pump-freezing.md.

What the lead received from each researcher, after the guard's check:
  - Some air-source heat pumps can operate below 0°F, and some do so without electric-resistance backup. — "Thes…
     … CHECKED: 3 claim(s) kept, 0 dropped (no quote or url).
  - EV battery packs typically take 10 years or more and well over 100,000 miles before starting to lose “seriou…
     … CHECKED: 2 claim(s) kept, 0 dro

In [10]:
verdict = next((m.text for m in messages if isinstance(m, ToolMessage) and m.name == "task" and "VERDICT" in m.text), "")
print("Verifier:\n" + verdict[-1500:])

brief = messages[-1].text
(OUT / "brief.md").write_text(brief, encoding="utf-8")
print("\n" + "=" * 80 + "\n" + brief + f"\n\nsaved: {OUT / 'brief.md'}")

Verifier:
Some air-source heat pumps can operate below 0°F, and many do so without electric-resistance backup. → supported (https://www1.eere.energy.gov/buildings/publications/pdfs/building_america/inverter-driven-heat-pumps-cold.pdf)
Traditional heat pumps may suffer performance decline in colder climates, while cold-climate heat pumps are designed to operate with greater capacity and efficiency below 32°F. → supported (https://www.energy.gov/cmei/buildings/residential-cold-climate-heat-pump-challenge)
EV battery packs typically take 10 years or more and well over 100,000 miles before starting to lose “serious muscle.” → supported (https://afdc.energy.gov/case/2788?text=)
Geotab’s 2024 study found average EV battery degradation of 1.8% per year. → supported (https://www.geotab.com/blog/ev-battery-health)
“None found” under Conflicts. → supported (evidence files contain no conflicting statements)
Rooftop-solar payback: the researcher failed to retrieve evidence, so no reliable payback 

## 7. Timeout: a researcher that never returns

One sub-question, and a search that hangs. The guard cuts the `task` call after `TASK_TIMEOUT` seconds; the lead
answers with a gap instead of waiting forever, and does not retry (a rule in `prompts/lead.md`).

In [11]:
FAULTS.update(hang="heat pump")
state, usage, seconds = await stream(lead, "Brief with one sub-question: how well do heat pumps work below 0°C?",
                                     main="lead", show_tokens=False)
FAULTS.update(hang=None)
print(state["messages"][-1].text)
usage.table(f"Timeout run: {seconds:.0f} s (guard timeout {TASK_TIMEOUT} s)")

[lead] → task({"description": "Objective: Research how well heat pumps work below 0°C, focusing on perfo…)
[researcher] → search({"query": "site:energy.gov heat pump cold climates below zero performance efficiency cold …)


WARNING | subagent researcher failed: timed out after 60 s


[lead] ← task: SUBAGENT FAILED (timed out after 60 s). Report this sub-question under Gaps; do not invent…

## Findings
None.

## Conflicts
None found.

## Gaps
- How well heat pumps work below 0°C: the research subagent timed out, so no reliable findings are available.

Timeout run: 64 s (guard timeout 60 s)
agent                    model calls  input tok output tok
lead                               2      4,920        155
researcher                         1      1,884        149
total                              3      6,804        304
